In [1]:
# 2) Load a small Qwen model + tokenizer
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"  # small instruct variant

# Device/dtype setup
if torch.cuda.is_available():
    device = "cuda"
    dtype = torch.float16
elif torch.backends.mps.is_available():
    device = "mps"
    dtype = torch.float16
else:
    device = "cpu"
    dtype = torch.float32

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype)
model.to(device)
model.eval()

print(f"Loaded {model_name} on {device} with dtype={dtype}")

/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/conda/lib/python3.10/site-packages/transformers/utils/hub.py:106: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loaded Qwen/Qwen2.5-0.5B-Instruct on cuda with dtype=torch.float16


In [2]:
# --- High-level summary ---
cfg = model.config

print("=== Config summary ===")
print(f"model_type: {getattr(cfg, 'model_type', None)}")
print(f"hidden_size: {getattr(cfg, 'hidden_size', None)}")
print(f"num_hidden_layers: {getattr(cfg, 'num_hidden_layers', None)}")
print(f"num_attention_heads: {getattr(cfg, 'num_attention_heads', None)}")
print(f"num_key_value_heads (GQA): {getattr(cfg, 'num_key_value_heads', None)}")
print(f"vocab_size: {getattr(cfg, 'vocab_size', None)}")
print(f"rope_theta: {getattr(cfg, 'rope_theta', None)}")
print(f"rope_scaling: {getattr(cfg, 'rope_scaling', None)}")
print(f"rms_norm_eps: {getattr(cfg, 'rms_norm_eps', None)}")
print(f"max_position_embeddings: {getattr(cfg, 'max_position_embeddings', None)}")

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("\n=== Parameters ===")
print(f"total params: {total_params:,}")
print(f"trainable:   {trainable_params:,}")
print(f"precision:   {dtype}, device: {device}")

=== Config summary ===
model_type: qwen2
hidden_size: 896
num_hidden_layers: 24
num_attention_heads: 14
num_key_value_heads (GQA): 2
vocab_size: 151936
rope_theta: 1000000.0
rope_scaling: None
rms_norm_eps: 1e-06
max_position_embeddings: 32768

=== Parameters ===
total params: 494,032,768
trainable:   494,032,768
precision:   torch.float16, device: cuda


In [19]:
# --- Shallow module tree ---
        
def print_tree(mod, max_depth=2, prefix=""):
    if max_depth < 0:
        return

    has_child = False
    for name, child in mod.named_children():  # registered submodules only
        has_child = True
        print(f"{prefix}{name}: {child.__class__.__name__}")
        print_tree(child, max_depth - 1, prefix + "  ")

    if not has_child:
        # Leaf module: show local params/buffers (no recursion)
        for pname, p in mod.named_parameters(recurse=False):
            print(f"{prefix}- param {pname}: shape={tuple(p.shape)}, dtype={p.dtype}")
        for bname, b in mod.named_buffers(recurse=False):
            print(f"{prefix}- buffer {bname}: shape={tuple(b.shape)}, dtype={b.dtype}")

print("=== Top-level module tree (depth<=2) ===")
print_tree(model, max_depth=6)

=== Top-level module tree (depth<=2) ===
model: Qwen2Model
  embed_tokens: Embedding
    - param weight: shape=(151936, 896), dtype=torch.float16
  layers: ModuleList
    0: Qwen2DecoderLayer
      self_attn: Qwen2Attention
        q_proj: Linear
          - param weight: shape=(896, 896), dtype=torch.float16
          - param bias: shape=(896,), dtype=torch.float16
        k_proj: Linear
          - param weight: shape=(128, 896), dtype=torch.float16
          - param bias: shape=(128,), dtype=torch.float16
        v_proj: Linear
          - param weight: shape=(128, 896), dtype=torch.float16
          - param bias: shape=(128,), dtype=torch.float16
        o_proj: Linear
          - param weight: shape=(896, 896), dtype=torch.float16
      mlp: Qwen2MLP
        gate_proj: Linear
          - param weight: shape=(4864, 896), dtype=torch.float16
        up_proj: Linear
          - param weight: shape=(4864, 896), dtype=torch.float16
        down_proj: Linear
          - param weight: sh

In [ ]:
# --- Locate attention + RoPE (robust to naming/implementation) ---
base = getattr(model, "model", model)  # Qwen2 uses model.model for the backbone
layers = getattr(base, "layers", None) # get list of transformer layers
assert layers is not None and len(layers) > 0, "Could not find transformer layers."

# Handle both .self_attn and .attn naming
attn = None
layer_idx = None
for i, layer in enumerate(layers):
    for attr in ("self_attn", "attn"):
        if hasattr(layer, attr):
            attn = getattr(layer, attr)
            layer_idx = i
            break
    if attn is not None:
        break

assert attn is not None, "Could not find an attention module in the first few layers."

# Define a RoPE finder that’s resilient to naming differences
def _find_rope_in(module):
    # 1) direct attributes commonly used
    for name in ("rotary_emb", "rope", "rotary"):
        if hasattr(module, name):
            obj = getattr(module, name)
            if obj is not None:
                return obj, f"{module.__class__.__name__}.{name}"
    # 2) nested submodules that look like RoPE
    for name, sub in module.named_modules():
        cls = sub.__class__.__name__.lower()
        if "rotary" in cls or "rope" in cls or hasattr(sub, "inv_freq") or hasattr(sub, "cos_cached"):
            return sub, f"{module.__class__.__name__}.{name}"
    return None, None

# Try to find RoPE in attention, then anywhere in the model
rope, where = _find_rope_in(attn)
if rope is None:
    # try anywhere in the model
    print("RoPE not found in attention; searching entire model...")
    rope, where = _find_rope_in(model)
    

if rope is None:
    # Construct a compatible rotary embedding as a fallback for inspection purposes
    print("No rotary module found on attention/model; constructing a compatible fallback.")
    try:
        from transformers.models.qwen2.modeling_qwen2 import Qwen2RotaryEmbedding as RopeImpl
    except Exception:
        from transformers.models.llama.modeling_llama import LlamaRotaryEmbedding as RopeImpl

    cfg = model.config
    num_heads = getattr(attn, "num_heads", getattr(cfg, "num_attention_heads", None))
    head_dim = getattr(attn, "head_dim", (cfg.hidden_size // num_heads) if num_heads else None)
    assert head_dim is not None, "Could not infer head_dim."

    # Try common constructor signatures across HF versions
    try:
        rope = RopeImpl(
            dim=head_dim,
            max_position_embeddings=getattr(cfg, "max_position_embeddings", 4096),
            base=getattr(cfg, "rope_theta", 10000.0),
            rope_scaling=getattr(cfg, "rope_scaling", None),
        )
    except TypeError:
        try:
            rope = RopeImpl(
                head_dim,
                max_position_embeddings=getattr(cfg, "max_position_embeddings", 4096),
                rope_theta=getattr(cfg, "rope_theta", 10000.0),
                rope_scaling=getattr(cfg, "rope_scaling", None),
            )
        except TypeError:
            rope = RopeImpl(head_dim)
    where = "constructed fallback"

print(f"=== First attention module (layer {layer_idx}) ===")
print(attn)
print("\n=== Rotary embedding source ===")
print(f"{where} -> {rope.__class__.__name__}")

# Try to read common RoPE internals
inv_freq = getattr(rope, "inv_freq", None)
rope_dim = None
if inv_freq is not None:
    rope_dim = inv_freq.numel() * 2
    print(f"inv_freq shape: {tuple(inv_freq.shape)} -> implied rotary dims: {rope_dim}")
    print("first 8 inv_freq values:", inv_freq[:8].detach().float().cpu().numpy())

for attr in ["theta", "rope_type", "scaling_type", "short_factor", "long_factor", "original_max_position_embeddings"]:
    if hasattr(rope, attr):
        print(f"{attr}: {getattr(rope, attr)}")

# Head geometry
cfg = model.config
num_heads = getattr(attn, "num_heads", getattr(cfg, "num_attention_heads", None))
head_dim = getattr(attn, "head_dim", (cfg.hidden_size // num_heads) if num_heads else None)
num_kv_heads = getattr(attn, "num_key_value_heads", getattr(cfg, "num_key_value_heads", None))
print(f"\nheads: {num_heads}, kv_heads: {num_kv_heads}, head_dim: {head_dim}, rotary_dim (approx): {rope_dim}")

=== First attention module (layer 0) ===
Qwen2Attention(
  (q_proj): Linear(in_features=896, out_features=896, bias=True)
  (k_proj): Linear(in_features=896, out_features=128, bias=True)
  (v_proj): Linear(in_features=896, out_features=128, bias=True)
  (o_proj): Linear(in_features=896, out_features=896, bias=False)
)

=== Rotary embedding source ===
Qwen2ForCausalLM.model.rotary_emb -> Qwen2RotaryEmbedding
inv_freq shape: (32,) -> implied rotary dims: 64
first 8 inv_freq values: [1.         0.64938164 0.4216965  0.27384198 0.17782794 0.1154782
 0.07498942 0.04869675]
rope_type: default

heads: 14, kv_heads: 2, head_dim: 64, rotary_dim (approx): 64


In [20]:
# --- New cell: show RoPE source code ---

import inspect, sys, os, re

cls = type(rope)
print(f"RoPE class: {cls.__module__}.{cls.__name__}")

# Try to retrieve the class source and where it lives
try:
    src = inspect.getsource(cls)
    file = inspect.getsourcefile(cls) or inspect.getfile(cls)
    start = inspect.getsourcelines(cls)[1]
    print(f"\nSource file: {file}:{start}\n")
    print(src)
except (OSError, TypeError):
    # Fallback: show module path and try to extract a nearby class snippet
    mod = sys.modules.get(cls.__module__)
    path = inspect.getfile(mod) if mod else None
    print(f"\nCould not get source directly. Module file: {path}")
    if path and os.path.exists(path):
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            code = f.read()
        m = re.search(rf"(?m)^class\s+{cls.__name__}\b.*?:", code)
        if m:
            window = code[m.start(): m.start() + 4000]  # show initial ~4KB after class def
            print("\n" + window)
        else:
            print("\nCouldn't locate class definition; showing lines mentioning 'rotary':\n")
            for i, line in enumerate(code.splitlines(), 1):
                if "rotary" in line.lower():
                    print(f"{i:5d}: {line}")

# Optionally, also show common helper functions from the same module (if present)
helpers = []
for name in ("apply_rotary_pos_emb", "apply_rotary_pos_emb_torch", "apply_rotary_embedding", "apply_rotary_emb"):
    fn = getattr(sys.modules.get(cls.__module__, object()), name, None)
    if fn and inspect.isfunction(fn):
        helpers.append((name, fn))

for name, fn in helpers:
    try:
        file = inspect.getsourcefile(fn) or inspect.getfile(fn)
        start = inspect.getsourcelines(fn)[1]
        print(f"\nHelper {name} at {file}:{start}\n")
        print(inspect.getsource(fn))
    except Exception as e:
        print(f"\nHelper {name}: could not retrieve source: {e}")

RoPE class: transformers.models.qwen2.modeling_qwen2.Qwen2RotaryEmbedding

Source file: /opt/conda/lib/python3.10/site-packages/transformers/models/qwen2/modeling_qwen2.py:286

class Qwen2RotaryEmbedding(nn.Module):
    def __init__(self, config: Qwen2Config, device=None):
        super().__init__()
        # BC: "rope_type" was originally "type"
        if hasattr(config, "rope_scaling") and config.rope_scaling is not None:
            self.rope_type = config.rope_scaling.get("rope_type", config.rope_scaling.get("type"))
        else:
            self.rope_type = "default"
        self.max_seq_len_cached = config.max_position_embeddings
        self.original_max_seq_len = config.max_position_embeddings

        self.config = config
        self.rope_init_fn = ROPE_INIT_FUNCTIONS[self.rope_type]

        inv_freq, self.attention_scaling = self.rope_init_fn(self.config, device)
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self.original_inv_freq = self.inv

In [9]:
# --- Demonstrate RoPE numerically on dummy Q/K (Qwen2 layout, robust call + fallback) ---
import torch

# Ensure attn/rope exist (lightweight recovery if not in scope)
base = getattr(model, "model", model)
layers = getattr(base, "layers", None)
if "attn" not in locals():
    attn = None
    if layers is not None:
        for lyr in layers:
            for attr in ("self_attn", "attn"):
                if hasattr(lyr, attr):
                    attn = getattr(lyr, attr)
                    break
            if attn is not None:
                break

rope = locals().get("rope", None)
if rope is None and attn is not None:
    for name in ("rotary_emb", "rope", "rotary"):
        if hasattr(attn, name):
            rope = getattr(attn, name)
            break

# Try to import the model's apply_rotary_pos_emb utility
try:
    from transformers.models.qwen2.modeling_qwen2 import apply_rotary_pos_emb
except Exception:
    from transformers.models.llama.modeling_llama import apply_rotary_pos_emb

cfg = model.config
num_heads = getattr(attn, "num_heads", getattr(cfg, "num_attention_heads", None))
head_dim = getattr(attn, "head_dim", getattr(cfg, "hidden_size", None) // num_heads)
bsz, seqlen = 1, 8

# Qwen2 apply_rotary_pos_emb expects (bsz, num_heads, seqlen, head_dim)
q = torch.randn(bsz, num_heads, seqlen, head_dim, device=device, dtype=next(model.parameters()).dtype)
k = torch.randn_like(q)
position_ids = torch.arange(seqlen, device=device).unsqueeze(0)  # (bsz, seqlen)

def get_cos_sin(rope, q, position_ids):
    # Preferred path: use the model's rope module if available
    if rope is not None:
        try:
            return rope(q, position_ids=position_ids)  # many HF impls ignore q's layout
        except Exception:
            pass
        try:
            return rope(q, position_ids)
        except Exception:
            pass

    # Manual fallback from inv_freq or config
    inv_freq = getattr(rope, "inv_freq", None) if rope is not None else None
    if inv_freq is None:
        rotary_dim = head_dim  # Qwen/LLaMA typically rotate full head_dim
        base = getattr(cfg, "rope_theta", 10000.0)
        inv_freq = 1.0 / (base ** (torch.arange(0, rotary_dim, 2, device=device, dtype=q.dtype) / rotary_dim))
    else:
        inv_freq = inv_freq.to(device=device, dtype=q.dtype)
        rotary_dim = inv_freq.numel() * 2

    # angles: (bsz, seqlen, rotary_dim//2) -> cos/sin: (bsz, seqlen, rotary_dim)
    angles = torch.einsum("bn,d->bnd", position_ids.to(q.dtype), inv_freq)
    emb = torch.cat([angles, angles], dim=-1)
    cos = emb.cos()  # (bsz, seqlen, rotary_dim)
    sin = emb.sin()
    return cos, sin

cos, sin = get_cos_sin(rope, q, position_ids)

# Apply rotation (explicitly set unsqueeze_dim=1 for (bsz, num_heads, seqlen, head_dim) layout)
q_rot, k_rot = apply_rotary_pos_emb(q, k, cos, sin, position_ids, unsqueeze_dim=1)

print("shapes -> q:", tuple(q.shape), "cos:", tuple(cos.shape))
print("q[0,0,0,:8] (before):", q[0,0,0,:8].detach().float().cpu().numpy())
print("q[0,0,0,:8] (after) :", q_rot[0,0,0,:8].detach().float().cpu().numpy())

# Optional: show angle matrix for first few rotary freqs (if available)
inv_freq = getattr(rope, "inv_freq", None) if rope is not None else None
if inv_freq is not None:
    d2 = min(8, inv_freq.numel())
    angles = torch.outer(position_ids[0].float().cpu(), inv_freq[:d2].float().cpu())  # (seqlen, d2)
    print("\nangle matrix (pos x first few freq):")
    print(angles)

shapes -> q: (1, 14, 8, 64) cos: (1, 8, 64)
q[0,0,0,:8] (before): [ 2.7128906  -1.2636719  -1.9892578  -2.6621094  -0.97509766 -0.1887207
  0.18322754  0.57421875]
q[0,0,0,:8] (after) : [ 2.7128906  -1.2636719  -1.9892578  -2.6621094  -0.97509766 -0.1887207
  0.18322754  0.57421875]

angle matrix (pos x first few freq):
tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [1.0000, 0.6494, 0.4217, 0.2738, 0.1778, 0.1155, 0.0750, 0.0487],
        [2.0000, 1.2988, 0.8434, 0.5477, 0.3557, 0.2310, 0.1500, 0.0974],
        [3.0000, 1.9481, 1.2651, 0.8215, 0.5335, 0.3464, 0.2250, 0.1461],
        [4.0000, 2.5975, 1.6868, 1.0954, 0.7113, 0.4619, 0.3000, 0.1948],
        [5.0000, 3.2469, 2.1085, 1.3692, 0.8891, 0.5774, 0.3749, 0.2435],
        [6.0000, 3.8963, 2.5302, 1.6431, 1.0670, 0.6929, 0.4499, 0.2922],
        [7.0000, 4.5457, 2.9519, 1.9169, 1.2448, 0.8083, 0.5249, 0.3409]])
